In [1]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import random
import numpy as np
import unidecode
import plotly.express as px
import plotly.graph_objects as go
from shapely import wkt


In [2]:
df_cim = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/CODECIM_updated.csv.xlsx")
df_cim.drop('Unnamed: 3', axis=1, inplace=True)
#df_patients = pd.read_csv("../data/data_cleaned/patients_FR_geocoded.csv",sep=";") #, dtype = {'adresse':str}).drop(["Unnamed: 0.1","Unnamed: 0"],axis=1)
df_clinique = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/pseudonymisation_id_sexe_ddn_loc.xlsx")
len(df_clinique)

df_patients = pd.read_csv("data/data_cleaned/patients_FR_metrop_geocoded.csv", sep=";")
len(df_patients)

C:\Users\lpokambo\AppData\Local\Temp\ipykernel_4152\2751263468.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_patients = pd.read_csv("data/data_cleaned/patients_FR_metrop_geocoded.csv", sep=";")


61911

## Data Cleaning 

### Adresses et geometries nulles: 

In [3]:

#df_patients= df_patients.dropna(subset=["adresse"])
#len(df_patients)
#df_patients['geometry'] = df_patients['geometry'].apply(wkt.loads)

# gdp_patients = gpd.GeoDataFrame(df_patients, geometry="geometry")
# len(df_patients[df_patients["pseudo_provisoire"].isna()])



### suppression de tous les doublons

In [4]:
df_clinique = df_clinique[df_clinique["cancernum"]==1]
len(df_clinique)

63881

In [5]:
count = df_clinique["pseudo_provisoire"].value_counts()
count
print(len(count[count==1]))

63881


In [6]:
df_clinique = df_clinique.dropna(subset=['pseudo_provisoire'])
df_patients = df_patients.dropna(subset=['pseudo_provisoire'])
len(df_clinique)

63881

In [7]:
df_patients

,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,...,address_has_num_init,address_has_num_geoloc,same_city,hostel,hosted,date_geoloc,geometry,CODE_IRIS,INSEE_REG,CODE_DEPT
0,0,0,1,34 RUE DES FRERES CHAUSSONS,92600.0,ASNIERES-SUR-SEINE,34 RUE DES FRERES CHAUSSONS 92600 ASNI...,2.289499,48.916298,0.832367,...,True,True,True,False,False,2024-01-05,POINT (647928.3162985359 6868712.859312387),920040302,11.0,92
1,1,1,2,11 RUE EMILE DUBOIS,75014.0,PARIS,11 RUE EMILE DUBOIS 75014 PARIS,2.336628,48.831707,0.972567,...,True,True,True,False,False,2024-01-05,POINT (651303.232633862 6859276.896689738),751145406,11.0,75
2,2,2,3,48 CHEMIN VERT,78680.0,EPONE,48 CHEMIN VERT 78680 EPONE,1.797376,48.950412,0.960861,...,True,True,True,False,False,2024-01-05,POINT (611921.2622170823 6872942.907671936),782170102,11.0,78
3,3,3,4,18 ALLEE DE LA CHARNILLE,47140.0,SAINT-SYLVESTRE-SUR-LOT,18 ALLEE DE LA CHARNILLE 47140 SAIN...,0.809474,44.404892,0.805540,...,True,True,True,False,False,2024-01-05,POINT (525576.5290254143 6369736.268092031),472800000,75.0,47
4,4,4,5,31 RUE DU GENERAL DE MIRIBEL,92500.0,RUEIL-MALMAISON,31 RUE DU GENERAL DE MIRIBEL 92500 RUEI...,2.173326,48.865232,0.973612,...,True,True,True,False,False,2024-01-05,POINT (639354.9912859926 6863117.583096649),920630504,11.0,92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61906,64289,64289,64290,3 AV DE FOUILLEUSE,92210,SAINT-CLOUD,3 AV DE FOUILLEUSE 92210 SAIN...,2.211229,48.861208,0.749249,...,True,True,True,False,False,2024-01-05,POINT (642131.0004121892 6862641.7109631905),920640101,11.0,92
61907,64290,64290,64291,81 COTE DU TORCHON,27220,L'HABIT,81 COTE DU TORCHON 27220 L'HABIT,1.365413,48.871685,0.946441,...,True,True,True,False,False,2024-01-05,POINT (580107.5206945667 6864758.656890908),273090000,28.0,27
61908,64291,64291,64292,60 RUE BAUDRICOURT,75013,Paris 13,60 RUE BAUDRICOURT 75013 Pari...,2.362960,48.825882,0.811235,...,True,True,True,False,False,2024-01-05,POINT (653230.9422174117 6858613.30609641),751135006,11.0,75
61909,64292,64292,64293,159 AVENUE DE LA REPUBLIQUE,92320,CHATILLON,159 AVENUE DE LA REPUBLIQUE 92320 CHAT...,2.300784,48.809219,0.970667,...,True,True,True,False,False,2024-01-05,POINT (648649.9241903964 6856799.224564967),920200101,11.0,92


In [8]:
df_adresse_clinique = df_patients.merge(df_clinique, on='pseudo_provisoire', how='left')
#df_patients_new_cim = df_adresse_clinique.merge(df_cim, on='topo_initiale_cim10', how='left')
print(df_adresse_clinique)

       Unnamed: 0.1  Unnamed: 0  pseudo_provisoire  \
0                 0           0                  1   
1                 1           1                  2   
2                 2           2                  3   
3                 3           3                  4   
4                 4           4                  5   
...             ...         ...                ...   
61906         64289       64289              64290   
61907         64290       64290              64291   
61908         64291       64291              64292   
61909         64292       64292              64293   
61910         64293       64293              64294   

                                                 adresse codepost  \
0                    34 RUE DES FRERES CHAUSSONS          92600.0   
1                    11 RUE EMILE DUBOIS                  75014.0   
2                    48 CHEMIN VERT                       78680.0   
3                    18 ALLEE DE LA CHARNILLE             47140.0   
4     

In [9]:
len(df_adresse_clinique)

61911

## Age Classes

In [10]:
len(df_adresse_clinique)

61911

In [11]:
id_dcd = pd.read_excel('../geocodeur/from_hegp/data/data_octobre_2023/dcd_pseudo.xlsx')
dcd = id_dcd['pseudo_provisoire'].to_list()

df_adresse_clinique_non_dcd = df_adresse_clinique[~df_adresse_clinique['pseudo_provisoire'].isin(dcd)]
len(df_adresse_clinique_non_dcd)

61743

In [12]:
df_adresse_clinique_no_age = df_adresse_clinique_non_dcd[~pd.isna(df_adresse_clinique_non_dcd['ageaudiag'])]

len(df_adresse_clinique_no_age)

61344

In [13]:

df_adulte = df_adresse_clinique_no_age[df_adresse_clinique_no_age["ageaudiag"]>=18]
df_enfant = df_adresse_clinique_no_age[df_adresse_clinique_no_age["ageaudiag"]<18]

In [14]:
len(df_enfant)

3466

In [15]:
df_enfant.describe()
#print(len(df_adulte))
#df_adulte.to_csv("H:/canc_air/data/data_cleaned/patients_FR_geocoded_adulte_clinique.csv",sep=";")

,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,x,y,score,pc_city,INSEE_REG,ageaudiag,cancernum
count,3466.000000,3466.000000,3466.000000,3466.000000,3466.000000,3466.000000,3466.000000,3466.000000,3466.000000,3466.0
mean,34309.452683,34309.452683,34310.452683,2.324451,48.051357,0.783694,67369.761396,30.315638,4.953260,1.0
std,17810.347143,17810.347143,17810.347143,1.911189,1.743989,0.197734,26117.741435,26.495504,5.255175,0.0
min,10.000000,10.000000,11.000000,-4.603458,41.909856,0.121804,1130.000000,11.000000,0.000000,1.0
25%,20008.750000,20008.750000,20009.750000,2.034903,47.895658,0.629723,49230.000000,11.000000,1.000000,1.0
50%,34690.000000,34690.000000,34691.000000,2.341258,48.830751,0.816084,75019.000000,11.000000,3.000000,1.0
75%,49743.000000,49743.000000,49744.000000,2.627256,48.923906,0.962181,92100.000000,44.000000,9.000000,1.0
max,64286.000000,64286.000000,64287.000000,9.486877,51.075329,0.986250,95880.000000,94.000000,17.000000,1.0


In [16]:
#df_adresse_clinique["ageaudiag"].isna().sum()
#df_adresse_clinique["date_naissance"].isna().sum()

#len(df_adulte)+
#len(df_enfant) #+
df_adresse_clinique["ageaudiag"].isna().sum()
df_adulte = df_adulte.dropna(subset=["ageaudiag"])
len(df_adulte)

57878

In [17]:
#df_adulte.to_csv("C:/Users/lpokambo/Desktop/PROJET/projet_loice/data/sorties/excels/patients_FR_geocoded_adultes_clinique.csv", sep=";")

## Gender 

In [18]:
df_H = df_adulte[df_adulte["patient_sexe"]=="M"]
df_F = df_adulte[df_adulte["patient_sexe"]=="F"]

In [19]:
len(df_H)

13001

In [20]:
len(df_F)

44877

In [21]:
df_adulte[df_adulte['ageaudiag']>100]

,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,...,INSEE_REG,CODE_DEPT,patient_sexe,date_naissance,centre,ageaudiag,cancernum,date_diag,topo_initiale_cim10,topo_initialelib
41054,42502,42502,42503,5 RUE DE LA PLAINE,78770,MARCQ,5 RUE DE LA PLAINE 78770 MARCQ,1.825751,48.859078,0.945973,...,11.0,78,F,1900-01,saint-cloud,113.0,1.0,2013-09-10,C73,Tumeur maligne de la thyroïde
44364,45954,45954,45955,X13 BD DE L HOPITAL STELL,92500,RUEIL-MALMAISON,X13 BD DE L HOPITAL STELL 92500 RUEI...,2.186006,48.878955,0.682742,...,11.0,92,F,1900-01,saint-cloud,113.0,1.0,2013-06-27,C50,Tumeur maligne du quadrant supéro-externe du sein


In [22]:
df_F.describe()

,Unnamed: 0.1,Unnamed: 0,pseudo_provisoire,x,y,score,pc_city,INSEE_REG,ageaudiag,cancernum
count,44877.000000,44877.000000,44877.000000,44877.000000,44877.000000,44877.000000,44877.000000,44877.000000,44877.000000,44877.0
mean,31884.696459,31884.696459,31885.696459,2.215792,48.589651,0.843705,76893.466364,17.750162,55.235911,1.0
std,18576.610614,18576.610614,18576.610614,1.128953,1.022996,0.155164,20316.790613,16.760761,13.659123,0.0
min,0.000000,0.000000,1.000000,-4.777221,41.387744,0.105581,1170.000000,11.000000,18.000000,1.0
25%,15525.000000,15525.000000,15526.000000,2.144878,48.763779,0.761943,75013.000000,11.000000,45.000000,1.0
50%,31793.000000,31793.000000,31794.000000,2.288610,48.841715,0.948222,78340.000000,11.000000,55.000000,1.0
75%,47955.000000,47955.000000,47956.000000,2.406216,48.894289,0.966295,92290.000000,11.000000,65.000000,1.0
max,64293.000000,64293.000000,64294.000000,9.495744,51.050136,0.991167,95880.000000,94.000000,113.000000,1.0


## Localisation

In [23]:
df_adulte.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'pseudo_provisoire', 'adresse',
       'codepost', 'nom_commune_postal', 'requete', 'x', 'y', 'score',
       'trust_score', 'street', 'city', 'pc_city', 'ic_city', 'code_dept',
       'dept', 'reg', 'address', 'address_has_num_init',
       'address_has_num_geoloc', 'same_city', 'hostel', 'hosted',
       'date_geoloc', 'geometry', 'CODE_IRIS', 'INSEE_REG', 'CODE_DEPT',
       'patient_sexe', 'date_naissance', 'centre', 'ageaudiag', 'cancernum',
       'date_diag', 'topo_initiale_cim10', 'topo_initialelib'],
      dtype='object')

## Patho

In [24]:
df_cim = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/CODECIM_updated.csv.xlsx")
df_cim.drop('Unnamed: 3', axis=1, inplace=True)

In [25]:
correspondance = df_cim.set_index('topo_initiale_cim10')['Proposition_Clémence_1'].to_dict()


In [26]:

df_adulte['patho'] = df_adulte['topo_initiale_cim10'].map(correspondance)

df_adulte.columns
df_adulte.drop(columns =["Unnamed: 0.1","Unnamed: 0"], axis=1, inplace=True)


In [61]:
df_adulte.columns

Index(['Unnamed: 0', 'pseudo_provisoire', 'adresse', 'codepost',
       'nom_commune_postal', 'requete', 'x', 'y', 'score', 'trust_score',
       'street', 'city', 'pc_city', 'ic_city', 'code_dept', 'dept', 'reg',
       'address', 'address_has_num_init', 'address_has_num_geoloc',
       'same_city', 'hostel', 'hosted', 'date_geoloc', 'geometry', 'CODE_IRIS',
       'INSEE_REG', 'CODE_DEPT', 'patient_sexe', 'date_naissance', 'centre',
       'ageaudiag', 'cancernum', 'date_diag', 'topo_initiale_cim10',
       'topo_initialelib', 'patho'],
      dtype='object')

In [69]:
dep_idf = ["75","77","78","91","92","93","94","95"]
df_adulte_idf = df_adulte[df_adulte['INSEE_REG']==11]
print(len(df_adulte_idf))

44771


In [67]:
df_patients_co = df_adulte_idf.groupby("CODE_DEPT").size().reset_index(name= 'patient_co')
df_patients_co


,CODE_DEPT,patient_co
0,60,1
1,75,10892
2,77,3018
3,78,8628
4,91,2992
5,92,10174
6,93,3204
7,94,3038
8,95,2824


In [68]:
df_adulte_idf = df_adulte_idf[df_adulte_idf['CODE_DEPT'].isin(dep_idf)]
len(df_adulte_idf)

44770

In [70]:
patiente = df_adulte_idf[df_adulte_idf["CODE_DEPT"]=='60']
patiente

,Unnamed: 0,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,trust_score,...,CODE_DEPT,patient_sexe,date_naissance,centre,ageaudiag,cancernum,date_diag,topo_initiale_cim10,topo_initialelib,patho
31505,33574,34710,48 RUE DE MONTMELIAN,95470,SAINT-WITZ,48 RUE DE MONTMELIAN 95470 SAIN...,2.574523,49.093371,0.955967,high,...,60,F,1981-01,saint-cloud,35.0,1.0,2016-10-20,C50,Tumeur maligne du quadrant supéro-externe du sein,Sein


In [29]:
df_adulte_idf = df_adulte_idf.merge(df_patients_co, on='CODE_DEPT')
#df_adulte_idf.drop(columns=['patient_co_y'],  axis=1, inplace=True)
#df_adulte_idf = df_adulte_idf.rename(columns={'patient_co_x': 'patient_co'}, inplace=True)
df_adulte_idf["CODE_DEPT"].unique()

array(['92', '75', '78', '95', '93', '91', '77', '94', '60'], dtype=object)

In [30]:
df_adulte.to_csv("Resultats/Excels/patients_FR_geocoded_adulte_clinique_patho.csv",sep=";")
df_adulte_idf.to_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_patho.csv", sep=";")

In [31]:
df_adulte_idf_counts = df_adulte_idf[['CODE_DEPT','patient_co']].drop_duplicates(subset=['CODE_DEPT'])
df_adulte_idf_counts.sort_values(by='CODE_DEPT')
df_adulte_idf_counts

,CODE_DEPT,patient_co
0,92,10174
1,75,10892
2,78,8628
10,95,2824
12,93,3204
16,91,2992
20,77,3018
46,94,3038
24679,60,1


In [32]:
df_adulte_idf_counts.to_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_counts.csv", sep=";")

In [33]:
df_patho_count = pd.DataFrame(df_adulte['patho'].value_counts().reset_index(), columns=["patho","count"])
label = df_patho_count["patho"]
values = df_patho_count["count"]


In [34]:

df_patho_count = df_patho_count.rename(columns={'patho':'Pathologie'})


In [35]:
df_patho_count['count'].sum()

57878

In [36]:
df_adulte[df_adulte['topo_initiale_cim10'].isna()]

,pseudo_provisoire,adresse,codepost,nom_commune_postal,requete,x,y,score,trust_score,street,...,CODE_DEPT,patient_sexe,date_naissance,centre,ageaudiag,cancernum,date_diag,topo_initiale_cim10,topo_initialelib,patho


In [37]:
fig = go.Figure(data=[go.Pie(labels=df_patho_count["Pathologie"], values=df_patho_count["count"])])
fig.show()

## Treatment during the period 

In [38]:
df_adulte = pd.read_csv("Resultats/Excels/patients_FR_geocoded_adulte_clinique_patho.csv",sep=";", dtype={"codepost":str})
id_dcd = pd.read_excel('../geocodeur/from_hegp/data/data_octobre_2023/dcd_pseudo.xlsx')
dcd = id_dcd['pseudo_provisoire'].to_list()
df_adulte = df_adulte[~df_adulte['pseudo_provisoire'].isin(dcd)]
len(df_adulte)


57878

In [39]:
df_chimio = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/chimio.xlsx")
df_radio = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/radiothérapie.xlsx")
df_chirurgie = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/chirurgie.xlsx")
df_oraux = pd.read_excel("../geocodeur/from_hegp/data/data_octobre_2023/w_ttt_pseudo.xlsx")


In [40]:
len(df_oraux)

446

In [41]:
df_chimio_test = df_chimio.drop_duplicates(subset="pseudo_provisoire", keep="first")
df_radio_test = df_radio.drop_duplicates(subset="pseudo_provisoire", keep="first")
df_chirurgie_test = df_chirurgie.drop_duplicates(subset="pseudo_provisoire", keep="first")
df_oraux = df_oraux.drop_duplicates(subset="pseudo_provisoire", keep="first")

id_c = df_chimio_test["pseudo_provisoire"].unique()
id_r = df_radio_test["pseudo_provisoire"].unique()
id_ch= df_chirurgie_test["pseudo_provisoire"].unique()
id_or = df_oraux["pseudo_provisoire"].unique()
id_all_patients = df_adulte["pseudo_provisoire"].unique()

df_chimio_id_valid = df_adulte[df_adulte["pseudo_provisoire"].isin(id_c)]
df_radio_id_valid = df_adulte[df_adulte["pseudo_provisoire"].isin(id_r)]
df_chirurgie_id_valid = df_adulte[df_adulte["pseudo_provisoire"].isin(id_ch)]
df_oraux_id_valid = df_adulte[df_adulte["pseudo_provisoire"].isin(id_or)]

id_chimio = df_chimio_id_valid["pseudo_provisoire"].unique()
id_radio = df_radio_id_valid["pseudo_provisoire"].unique()
id_chirurgie = df_chirurgie_id_valid["pseudo_provisoire"].unique()
id_oraux = df_oraux_id_valid["pseudo_provisoire"].unique()


In [42]:
## Verification
# print(len(df_chimio_test[~df_chimio_test["pseudo_provisoire"].isin(id_chimio)]))
# print(len(df_radio_test[~df_radio_test["pseudo_provisoire"].isin(id_radio)]))
# print(len(df_chirurgie_test[~df_chirurgie_test["pseudo_provisoire"].isin(id_chirurgie)]))


 

In [43]:
print(len(df_adulte[df_adulte["pseudo_provisoire"].isin(id_chimio)]))
print(len(id_chimio))

print(len(df_adulte[df_adulte["pseudo_provisoire"].isin(id_radio)]))
print(len(id_radio))

print(len(df_adulte[df_adulte["pseudo_provisoire"].isin(id_chirurgie)]))
print(len(id_chirurgie))

print(len(df_adulte[df_adulte["pseudo_provisoire"].isin(id_oraux)]))
print(len(id_oraux))

10228
10228
13038
13038
12634
12634
446
446


In [44]:
# Conversion des vecteurs en ensembles
chimio = set(id_chimio)
radio = set(id_radio)
chirurgie = set(id_chirurgie)
oraux = set(id_oraux)
all_patients = set(id_all_patients)

# Debug: Print initial set sizes
print(f"Total patients: {len(all_patients)}")
print(f"Chimio: {len(chimio)}")
print(f"Radio: {len(radio)}")
print(f"Chirurgie: {len(chirurgie)}")
print(f"oraux: {len(oraux)}")

# Patients traités par un seul traitement 
uniquement_chimio = chimio - (radio | chirurgie | oraux)
uniquement_radio = radio - (chimio | chirurgie | oraux)
uniquement_chirurgie = chirurgie - (chimio | radio | oraux)
uniquement_oraux = oraux - (chimio | radio| chirurgie)

# Debug: Print sizes after calculating one treatment
print(f"Uniquement Chimio: {len(uniquement_chimio)}")
print(f"Uniquement Radio: {len(uniquement_radio)}")
print(f"Uniquement Chirurgie: {len(uniquement_chirurgie)}")
print(f"Uniquement Oraux: {len(uniquement_oraux)}")


# Patients traités avec une combinaison de deux traitements 
chirurgie_chimio = (chirurgie & chimio) - (uniquement_chirurgie | uniquement_chimio | oraux | radio)
radio_chirurgie = (radio & chirurgie) - (uniquement_radio | uniquement_chirurgie | chimio | oraux)
chimio_radio = (chimio & radio) - (uniquement_chimio | uniquement_radio | chirurgie | oraux)
chirurgie_oraux = (chirurgie & chimio) - (uniquement_chirurgie | uniquement_oraux | radio | chimio)
chimio_oraux = (chimio & oraux) - (uniquement_chimio | uniquement_oraux | chirurgie | radio)
radio_oraux = (oraux & radio) - (uniquement_oraux | uniquement_radio | chirurgie | chimio)

# Debug: Print sizes after calculating two treatments
print(f"Chirurgie Chimio: {len(chirurgie_chimio)}")
print(f"Radio Chirurgie: {len(radio_chirurgie)}")
print(f"Chimio Radio: {len(chimio_radio)}")
print(f"chirurgie_oraux: {len(chirurgie_oraux)}")
print(f"chimio_oraux: {len(chimio_oraux)}")
print(f"radio_oraux: {len(radio_oraux)}")


# Patients traités avec l'ensemble des traitements 
all_treatments = radio & chirurgie & chimio & oraux - ( (radio & chirurgie) | (chirurgie & chimio) | (radio & chimio) | (oraux & chimio) | (radio & oraux) | (oraux & chirurgie))
all_treatment = radio & chirurgie & chimio & oraux

print(f"All Treatments: {len(all_treatments)}")


# Patients traités par aucun traitement 
no_treatments = all_patients - ( all_treatments | chirurgie_chimio | radio_chirurgie | chimio_radio | uniquement_chimio | uniquement_radio | uniquement_chirurgie | chirurgie_oraux |chimio_oraux |radio_oraux |uniquement_oraux)

print(f"No Treatments: {len(no_treatments)}")


# Final check
total_count = len(uniquement_chimio) + len(uniquement_radio) + len(uniquement_chirurgie) + len(chirurgie_chimio) + len(radio_chirurgie) + len(chimio_radio) + len(all_treatments) + len(no_treatments)
print(f"Total Counted: {total_count}")

# Assert to verify
#assert total_count == len(all_patients), "Mismatch in patient counts"



Total patients: 57878
Chimio: 10228
Radio: 13038
Chirurgie: 12634
oraux: 446
Uniquement Chimio: 3213
Uniquement Radio: 4409
Uniquement Chirurgie: 3943
Uniquement Oraux: 345
Chirurgie Chimio: 1861
Radio Chirurgie: 3475
Chimio Radio: 1799
chirurgie_oraux: 0
chimio_oraux: 0
radio_oraux: 0
All Treatments: 0
No Treatments: 38833
Total Counted: 57533


In [45]:
print(len(no_treatments))
pd.Series(list(no_treatments)).to_csv("Resultats/Excels/no_treatments.csv",sep=";", index =False)
# f = open('no_treatments.txt','w')
# f.write(f"{no_treatments}")
# f.close()

38833


In [46]:
# Affichage des résultats
print("Uniquement traitement chimio:", len(uniquement_chimio),", soit : ", np.round(len(uniquement_chimio)/len(all_patients)*100, 2), "%")
print("Uniquement traitement radio:", len(uniquement_radio),", soit : ", np.round(len(uniquement_radio)/len(all_patients)*100, 2), "%")
print("Uniquement traitement chirurgie:", len(uniquement_chirurgie),", soit : ", np.round(len(uniquement_chirurgie)/len(all_patients)*100, 2) , "%")
print("Traitement chimio et radio:", len(chimio_radio),", soit : ", np.round(len(chimio_radio)/len(all_patients)*100, 2), "%")
print("Traitement radio et chirurgie:", len(radio_chirurgie),", soit : ", np.round(len(radio_chirurgie)/len(all_patients)*100, 2), "%")
print("Traitement chimio et chirurgie:", len(chirurgie_chimio),", soit : ", np.round(len(chirurgie_chimio)/len(all_patients)*100, 2), "%")
print("Traitement chimio, radio et chirurgie:", len(all_treatments),", soit : ", np.round(len(all_treatments)/len(all_patients)*100, 2), "%")
print("Traitement chimio, radio et chirurgie:", len(all_treatment),", soit : ", np.round(len(all_treatment)/len(all_patients)*100, 2), "%")

print("Sans traitement:", len(no_treatments), ", soit :", np.round(len(no_treatments)/len(all_patients)*100, 2), "%")

Uniquement traitement chimio: 3213 , soit :  5.55 %
Uniquement traitement radio: 4409 , soit :  7.62 %
Uniquement traitement chirurgie: 3943 , soit :  6.81 %
Traitement chimio et radio: 1799 , soit :  3.11 %
Traitement radio et chirurgie: 3475 , soit :  6.0 %
Traitement chimio et chirurgie: 1861 , soit :  3.22 %
Traitement chimio, radio et chirurgie: 0 , soit :  0.0 %
Traitement chimio, radio et chirurgie: 101 , soit :  0.17 %
Sans traitement: 38833 , soit : 67.09 %


In [47]:
# Conversion des vecteurs en ensembles
chimio = set(id_chimio)
radio = set(id_radio)
chirurgie = set(id_chirurgie)
all_patients = set(id_all_patients)

## & AND
## | OR 

# Patients traités avec l'ensemble des traitements 

all_treatments = radio & chirurgie & chimio - ( (radio & chirurgie) | (chirurgie & chimio) | (radio & chimio))

# Patients traités avec une combinaison de deux traitements 

chirurgie_chimio = chirurgie & chimio - all_treatments
radio_chirurgie = radio & chirurgie - all_treatments
chimio_radio = chimio & radio - all_treatments

# Patients traités par un seul traitement 

uniquement_chimio = chimio - (chirurgie_chimio | chimio_radio | all_treatments)
uniquement_radio = radio - (radio_chirurgie | chimio_radio | all_treatments)
uniquement_chirurgie = chirurgie - (radio_chirurgie | chirurgie_chimio | all_treatments)

# Patients traités par aucun traitement 

no_treatments = all_patients - ( all_treatments | chirurgie_chimio | radio_chirurgie | chimio_radio | uniquement_chimio | uniquement_radio | uniquement_chirurgie)


verif = len(all_treatments) + len(chirurgie_chimio) + len(radio_chirurgie) + len(chimio_radio) + len(uniquement_chimio) + len(uniquement_radio) + len(uniquement_chirurgie)
# # Patients traités uniquement avec chaque traitement
# uniquement_chimio = chimio - (radio | chirurgie)
# uniquement_radio = radio - (chimio | chirurgie)
# uniquement_chirurgie = chirurgie - (chimio | radio)
 

# # Patients traités avec des combinaisons de traitements
# chimio_radio = chimio & radio - chirurgie
# radio_chirurgie = radio & chirurgie - chimio
# chimio_chirurgie = chimio & chirurgie - radio

# patients_all_treatment = chimio & radio & chirurgie

# patients_all_treatment_exclusif = patients_all_treatment - (uniquement_chimio | uniquement_radio | uniquement_chirurgie |
#                                                             chimio_radio | radio_chirurgie | chimio_chirurgie)

# no_treatments = all_patients - (uniquement_chimio | uniquement_radio | uniquement_chirurgie |
#                                 chimio_radio | radio_chirurgie | chimio_chirurgie | patients_all_treatment_exclusif)

# Affichage des résultats
print("Uniquement traitement chimio:", len(uniquement_chimio),", soit : ", np.round(len(uniquement_chimio)/len(all_patients)*100, 2), "%")
print("Uniquement traitement radio:", len(uniquement_radio),", soit : ", np.round(len(uniquement_radio)/len(all_patients)*100, 2), "%")
print("Uniquement traitement chirurgie:", len(uniquement_chirurgie),", soit : ", np.round(len(uniquement_chirurgie)/len(all_patients)*100, 2) , "%")
print("Traitement chimio et radio:", len(chimio_radio),", soit : ", np.round(len(chimio_radio)/len(all_patients)*100, 2), "%")
print("Traitement radio et chirurgie:", len(radio_chirurgie),", soit : ", np.round(len(radio_chirurgie)/len(all_patients)*100, 2), "%")
print("Traitement chimio et chirurgie:", len(chirurgie_chimio),", soit : ", np.round(len(chirurgie_chimio)/len(all_patients)*100, 2), "%")
print("Traitement chimio, radio et chirurgie:", len(all_treatments),", soit : ", np.round(len(all_treatments)/len(all_patients)*100, 2), "%")
print("Sans traitement:", len(no_treatments), ", soit :", np.round(len(no_treatments)/len(all_patients)*100, 2), "%")

Uniquement traitement chimio: 3213 , soit :  5.55 %
Uniquement traitement radio: 4409 , soit :  7.62 %
Uniquement traitement chirurgie: 3943 , soit :  6.81 %
Traitement chimio et radio: 5154 , soit :  8.9 %
Traitement radio et chirurgie: 6830 , soit :  11.8 %
Traitement chimio et chirurgie: 5216 , soit :  9.01 %
Traitement chimio, radio et chirurgie: 0 , soit :  0.0 %
Sans traitement: 35823 , soit : 61.89 %


In [48]:
verif

28765